<a href="https://colab.research.google.com/github/knc6/jarvis-tools-notebooks/blob/master/jarvis-tools-notebooks/Timing_uMLFF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()

In [ ]:
%%time
!conda install dgl=2.1.0 chgnet matgl pytorch torchvision torchaudio pytorch-cuda -c pytorch -c nvidia -y --quiet

In [ ]:
%%time
import os
import torch
os.environ['TORCH'] = torch.__version__
print(torch.__version__)

!pip install -q torch-scatter -f https://data.pyg.org/whl/torch-${TORCH}.html
!pip install -q torch-sparse -f https://data.pyg.org/whl/torch-${TORCH}.html
!pip install -q git+https://github.com/pyg-team/pytorch_geometric.git

In [ ]:
%%time
!git clone https://github.com/ACEsuit/mace.git
!pip install -q ./mace

In [ ]:
%%time
!git clone https://github.com/MDIL-SNU/SevenNet.git
!pip install -q ./SevenNet/

In [ ]:
!pip install -q alignn

In [ ]:
import alignn
import matgl
import chgnet
import mace
import sevenn

In [ ]:
from alignn.ff.ff import AlignnAtomwiseCalculator, default_path, wt01_path, get_figshare_model_ff
from matgl.ext.ase import M3GNetCalculator
from sevenn.sevennet_calculator import SevenNetCalculator
from chgnet.model.dynamics import CHGNetCalculator
from mace.calculators import mace_mp
alignn_calculator = AlignnAtomwiseCalculator()

pot = matgl.load_model("M3GNet-MP-2021.2.8-PES")
matgl_calculator = M3GNetCalculator(pot)

sevenet_calculator = SevenNetCalculator("7net-0", device='cpu')

mace_calculator = mace_mp()

chgnet_calculator = CHGNetCalculator()

In [ ]:
# Dimer test
# import numpy as np
# a=np.arange(0.5,5,0.1)
# energies=[]
# for i in a:
#   atoms=Atoms(lattice_mat=[[20,0,0],[0,20,0],[0,0,20]],elements=['Al','Al'],coords=[[0,0,0],[0,0,i]],cartesian=True).ase_converter()
#   atoms.calc=alignn_calculator
#   print(atoms.get_potential_energy())
#   energies.append(atoms.get_potential_energy())
# import matplotlib.pyplot as plt
# %matplotlib inline
# plt.plot(a,energies)

In [ ]:
!pip install -q memory-profiler


In [ ]:
from ase.build.supercells import make_supercell
import numpy as np
import time
import matplotlib.pyplot as plt
from ase import Atoms,Atom
from ase.calculators.emt import EMT
from jarvis.io.vasp.inputs import Poscar
pos = """System
1.0
3.6 0.0 0.0
0.0 3.6 0.0
0.0 0.0 3.6
Cu
4
direct
0.0 0.0 0.0 Cu
0.0 0.5 0.5 Cu
0.5 0.0 0.5 Cu
0.5 0.5 0.0 Cu
"""
%matplotlib inline
emt_calc = EMT()
# a = 3.6
# atoms = Atoms([Atom('Cu', (0, 0, 0))],
#               cell=0.5 * a * np.array([[1.0, 1.0, 0.0],
#                                        [0.0, 1.0, 1.0],
#                                        [1.0, 0.0, 1.0]]),
#              pbc=True)
atoms=Poscar.from_string(pos).atoms.ase_converter()
numbers = np.arange(1,5,1)
times_emt = []
times_matgl = []
times_mace = []
times_chgnet = []
times_alignn = []
times_sevennet = []



natoms=[]
for i in numbers:
  print(i)
  sc=make_supercell(atoms, [[i,0,0],[0,i,0],[0,0,i]])

  t1=time.time()
  sc.calc = matgl_calculator
  en = sc.get_potential_energy()/len(sc)
  t2=time.time()
  times_matgl.append(t2-t1)

  t1=time.time()
  sc.calc = alignn_calculator
  en = sc.get_potential_energy()/len(sc)
  t2=time.time()
  times_alignn.append(t2-t1)

  t1=time.time()
  sc.calc = emt_calc
  en = sc.get_potential_energy()/len(sc)
  t2=time.time()
  times_emt.append(t2-t1)

  t1=time.time()
  sc.calc = mace_calculator
  en = sc.get_potential_energy()/len(sc)
  t2=time.time()
  times_mace.append(t2-t1)


  t1=time.time()
  sc.calc = chgnet_calculator
  en = sc.get_potential_energy()/len(sc)
  t2=time.time()
  times_chgnet.append(t2-t1)

  t1=time.time()
  sc.calc = sevenet_calculator
  en = sc.get_potential_energy()/len(sc)
  t2=time.time()
  times_sevennet.append(t2-t1)

  natoms.append(len(sc))

plt.plot(natoms,times_emt,'-o',label='EMT')
plt.plot(natoms,times_matgl,'-o',label='M3GNet')
plt.plot(natoms,times_alignn,'-o',label='ALIGNN')
plt.plot(natoms,times_mace,'-o',label='MACE')
plt.plot(natoms,times_chgnet,'-o',label='CHGNet')
plt.plot(natoms,times_sevennet,'-o',label='7Net')
plt.xlabel('Number of atoms')
plt.ylabel('Time (s)')
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
from memory_profiler import memory_usage
times_emt, times_matgl, times_mace, times_chgnet, times_alignn, times_sevennet = [], [], [], [], [], []
memory_emt, memory_matgl, memory_mace, memory_chgnet, memory_alignn, memory_sevennet = [], [], [], [], [], []

natoms = []
for i in numbers:
    print(i)
    sc = make_supercell(atoms, [[i, 0, 0], [0, i, 0], [0, 0, i]])

    # M3GNet
    sc.calc = matgl_calculator
    t1 = time.time()
    mem = memory_usage((sc.get_potential_energy,))
    t2 = time.time()
    times_matgl.append(t2 - t1)
    memory_matgl.append(max(mem) - min(mem))

    # ALIGNN
    sc.calc = alignn_calculator
    t1 = time.time()
    mem = memory_usage((sc.get_potential_energy,))
    t2 = time.time()
    times_alignn.append(t2 - t1)
    memory_alignn.append(max(mem) - min(mem))

    # EMT
    sc.calc = emt_calc
    t1 = time.time()
    mem = memory_usage((sc.get_potential_energy,))
    t2 = time.time()
    times_emt.append(t2 - t1)
    memory_emt.append(max(mem) - min(mem))

    # MACE
    sc.calc = mace_calculator
    t1 = time.time()
    mem = memory_usage((sc.get_potential_energy,))
    t2 = time.time()
    times_mace.append(t2 - t1)
    memory_mace.append(max(mem) - min(mem))

    # CHGNet
    sc.calc = chgnet_calculator
    t1 = time.time()
    mem = memory_usage((sc.get_potential_energy,))
    t2 = time.time()
    times_chgnet.append(t2 - t1)
    memory_chgnet.append(max(mem) - min(mem))

    # 7Net
    sc.calc = sevenet_calculator
    t1 = time.time()
    mem = memory_usage((sc.get_potential_energy,))
    t2 = time.time()
    times_sevennet.append(t2 - t1)
    memory_sevennet.append(max(mem) - min(mem))

    natoms.append(len(sc))

# Plot time scaling
plt.figure(figsize=(10, 5))
plt.plot(natoms, times_emt, '-o', label='EMT')
plt.plot(natoms, times_matgl, '-o', label='M3GNet')
plt.plot(natoms, times_alignn, '-o', label='ALIGNN')
plt.plot(natoms, times_mace, '-o', label='MACE')
plt.plot(natoms, times_chgnet, '-o', label='CHGNet')
plt.plot(natoms, times_sevennet, '-o', label='7Net')
plt.xlabel('Number of atoms')
plt.ylabel('Time (s)')
plt.grid(True)
plt.legend()
plt.title('Time Scaling Analysis')
plt.show()

# Plot memory usage
plt.figure(figsize=(10, 5))
plt.plot(natoms, memory_emt, '-o', label='EMT')
plt.plot(natoms, memory_matgl, '-o', label='M3GNet')
plt.plot(natoms, memory_alignn, '-o', label='ALIGNN')
plt.plot(natoms, memory_mace, '-o', label='MACE')
plt.plot(natoms, memory_chgnet, '-o', label='CHGNet')
plt.plot(natoms, memory_sevennet, '-o', label='7Net')
plt.xlabel('Number of atoms')
plt.ylabel('Memory Usage (MB)')
plt.grid(True)
plt.legend()
plt.title('Memory Usage Analysis')
plt.show()


In [ ]:
from ase.build.supercells import make_supercell
import numpy as np
import time
import matplotlib.pyplot as plt
from ase import Atoms,Atom
from ase.calculators.emt import EMT
from jarvis.io.vasp.inputs import Poscar
pos = """System
1.0
3.6 0.0 0.0
0.0 3.6 0.0
0.0 0.0 3.6
Cu
4
direct
0.0 0.0 0.0 Cu
0.0 0.5 0.5 Cu
0.5 0.0 0.5 Cu
0.5 0.5 0.0 Cu
"""
%matplotlib inline
# a = 3.6
# atoms = Atoms([Atom('Cu', (0, 0, 0))],
#               cell=0.5 * a * np.array([[1.0, 1.0, 0.0],
#                                        [0.0, 1.0, 1.0],
#                                        [1.0, 0.0, 1.0]]),
#              pbc=True)
atoms=Poscar.from_string(pos).atoms.ase_converter()
numbers = np.arange(1,9,1)

times_alignn = []




natoms=[]
for i in numbers:
  print(i)
  sc=make_supercell(atoms, [[i,0,0],[0,i,0],[0,0,i]])



  t1=time.time()
  sc.calc = alignn_calculator
  en = sc.get_potential_energy()/len(sc)
  t2=time.time()
  times_alignn.append(t2-t1)


  natoms.append(len(sc))


plt.plot(natoms,times_alignn,'-o',label='ALIGNN')

plt.xlabel('Number of atoms')
plt.ylabel('Time (s)')
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
from ase.build.supercells import make_supercell
import numpy as np
import time
import matplotlib.pyplot as plt
from ase import Atoms,Atom
from ase.calculators.emt import EMT
from jarvis.io.vasp.inputs import Poscar
pos = """System
1.0
3.6 0.0 0.0
0.0 3.6 0.0
0.0 0.0 3.6
Cu
4
direct
0.0 0.0 0.0 Cu
0.0 0.5 0.5 Cu
0.5 0.0 0.5 Cu
0.5 0.5 0.0 Cu
"""
%matplotlib inline
emt_calc = EMT()
# a = 3.6
# atoms = Atoms([Atom('Cu', (0, 0, 0))],
#               cell=0.5 * a * np.array([[1.0, 1.0, 0.0],
#                                        [0.0, 1.0, 1.0],
#                                        [1.0, 0.0, 1.0]]),
#              pbc=True)
atoms=Poscar.from_string(pos).atoms.ase_converter()
numbers = np.arange(1,9,1)
times_emt = []
times_matgl = []
times_alignn = []




natoms=[]
for i in numbers:
  print(i)
  sc=make_supercell(atoms, [[i,0,0],[0,i,0],[0,0,i]])

  t1=time.time()
  sc.calc = matgl_calculator
  en = sc.get_potential_energy()/len(sc)
  t2=time.time()
  times_matgl.append(t2-t1)

  t1=time.time()
  sc.calc = alignn_calculator
  en = sc.get_potential_energy()/len(sc)
  t2=time.time()
  times_alignn.append(t2-t1)

  t1=time.time()
  sc.calc = emt_calc
  en = sc.get_potential_energy()/len(sc)
  t2=time.time()
  times_emt.append(t2-t1)



  natoms.append(len(sc))

plt.plot(natoms,times_emt,'-o',label='EMT')
plt.plot(natoms,times_matgl,'-o',label='M3GNet')
plt.plot(natoms,times_alignn,'-o',label='ALIGNN')

plt.xlabel('Number of atoms')
plt.ylabel('Time (s)')
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
# from jarvis.db.figshare import data
# x=data('interfacedb')
# print(len(x))
# from jarvis.core.atoms import Atoms
# a=Atoms.from_dict(x[0]['atoms'])
# for i in  x:
#   # if 'JVASP-1002' in i['jid'] and 'JVASP-1174' in i['jid']:
#   if i['jid']=='Interface-JVASP-1002_JVASP-1174_film_miller_1_1_0_sub_miller_1_1_0_film_thickness_16_subs_thickness_16_seperation_2.5_disp_0.5_0.2_vasp':
#     a=Atoms.from_dict(i['atoms'])